# Эксперимент 2.4: Сравнение методов на ML задачах

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse

from oracles import REG_MODEL_NAMEL2Oracle, CLASS_MODEL_NAMEL2Oracle
from optimization import nonlinear_conjugate_gradients, hessian_free_newton, lbfgs

In [ ]:
# TODO: замени на загрузку своих LIBSVM датасетов из ЛР1.
rng = np.random.default_rng(42)
m, n = 4000, 300
A = rng.normal(size=(m, n))
A_sp = sparse.csr_matrix(A)
w = rng.normal(size=n)
b_reg = A @ w + 0.1 * rng.normal(size=m)
b_clf = np.sign(A @ w + 0.3 * rng.normal(size=m))
b_clf[b_clf == 0] = 1

def matvec_Ax(x):
    return A_sp.dot(x)
def matvec_ATx(y):
    return A_sp.T.dot(y)
def matmat_ATsA(s):
    return A_sp.T.dot(sparse.diags(s)).dot(A_sp).toarray()

regcoef = 1.0 / m
oracle_reg = REG_MODEL_NAMEL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b_reg, regcoef)
oracle_clf = CLASS_MODEL_NAMEL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b_clf, regcoef)

In [ ]:
def run_methods(oracle, x0):
    ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}
    out = {}
    out['NLCG'] = nonlinear_conjugate_gradients(oracle, x0, tolerance=1e-8, max_iter=150, line_search_options=ls, trace=True)
    out['HFN'] = hessian_free_newton(oracle, x0, tolerance=1e-8, max_iter=80, line_search_options=ls, trace=True)
    out['L-BFGS(L=10)'] = lbfgs(oracle, x0, tolerance=1e-8, max_iter=150, memory_size=10, line_search_options=ls, trace=True)
    return out

x0 = np.zeros(n)
res_reg = run_methods(oracle_reg, x0)
res_clf = run_methods(oracle_clf, x0)

In [ ]:
def draw_family(results, title):
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 3, 1)
    for name, (_, _, h) in results.items():
        plt.plot(h['func'], label=name)
    plt.xlabel('iteration')
    plt.ylabel('f(x_k)')
    plt.title(title + ': f vs iter')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 2)
    for name, (_, _, h) in results.items():
        plt.plot(h['time'], h['func'], label=name)
    plt.xlabel('time, sec')
    plt.ylabel('f(x_k)')
    plt.title(title + ': f vs time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(1, 3, 3)
    for name, (_, _, h) in results.items():
        g = np.array(h['grad_norm'])
        rel = (g ** 2) / max(g[0] ** 2, 1e-32)
        plt.plot(h['time'], rel, label=name)
    plt.yscale('log')
    plt.xlabel('time, sec')
    plt.ylabel(r'$||g_k||^2 / ||g_0||^2$')
    plt.title(title + ': rel grad vs time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.tight_layout()

draw_family(res_reg, 'Regression')
draw_family(res_clf, 'Classification')